# Import for bibliothek used for the analyse

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from datetime import date
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV
import ast
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np
import joblib
data = pd.read_csv("../data/immo_data_with_lat_lon.csv")
data.head()

,geo_krs,yearConstructed,baseRent,noRooms,heatingCosts,serviceCharge,livingSpace,noParkSpaces,balcony,hasKitchen,cellar,lift,garden,totalRent,full_address,adresseInLatLon
0,Dortmund,2018.0,972.60,3.0,43.05,215.00,87.00,1.0,True,False,True,True,False,1320.65,"Am_Dimberg 4 ,44229 Kirchhörde ,Nordrhein_West...","(51.457584985279, 7.455573960885)"
1,Göttingen_Kreis,2013.0,1343.48,5.0,160.00,290.00,127.95,1.0,True,False,True,True,False,1878.48,"Robert-Gernhardt-Platz 3 ,37073 Göttingen ,Nie...","(51.535960012831, 9.933195975578)"
2,Schwerin,2018.0,520.08,2.0,52.01,104.02,47.28,1.0,True,False,True,True,False,624.10,"Zum_Bahnhof 7 ,19055 Paulsstadt ,Mecklenburg_V...","(53.633845017121, 11.409782968187)"
3,Nordsachsen_Kreis,1930.0,240.00,2.0,60.00,40.00,44.66,2.0,False,False,False,False,False,340.00,"Turnerstraße 27 ,4435 Schkeuditz ,Sachsen","(51.398860407713, 12.219154626233)"
4,Main_Taunus_Kreis,2000.0,640.00,3.0,125.00,155.00,82.00,2.0,True,False,True,False,False,960.00,"Kurfürstenstr. 25 ,65439 Flörsheim_am_Main ,He...","(50.019472981812, 8.424950031762)"


# Add two columns contains Latitude and Longitude

In [2]:
data['adresseInLatLon'] = data['adresseInLatLon'].apply(ast.literal_eval)
data['latitude'] = data['adresseInLatLon'].apply(lambda x: x[0])
data['longitude'] = data['adresseInLatLon'].apply(lambda x: x[1])
#data.head()

# Add two more columns for the analysis depends of two columns

In [3]:
data = data[data["livingSpace"] > 0]
data[['yearConstructed','noParkSpaces','noRooms']]= data[['yearConstructed','noParkSpaces','noRooms']].astype(int)
data['age_of_house'] = date.today().year- data['yearConstructed']

data['price_per_m2'] = (data['baseRent']/ data['livingSpace']).round(4)

#  Drop not used columns

In [4]:
data.drop(['full_address', 'adresseInLatLon','geo_krs', 'heatingCosts','serviceCharge','baseRent'], axis='columns', inplace=True)
data.head()

,yearConstructed,noRooms,livingSpace,noParkSpaces,balcony,hasKitchen,cellar,lift,garden,totalRent,latitude,longitude,age_of_house,price_per_m2
0,2018,3,87.00,1,True,False,True,True,False,1320.65,51.457585,7.455574,7,11.1793
1,2013,5,127.95,1,True,False,True,True,False,1878.48,51.535960,9.933196,12,10.5000
2,2018,2,47.28,1,True,False,True,True,False,624.10,53.633845,11.409783,7,11.0000
3,1930,2,44.66,2,False,False,False,False,False,340.00,51.398860,12.219155,95,5.3739
4,2000,3,82.00,2,True,False,True,False,False,960.00,50.019473,8.424950,25,7.8049


# Encode all Boolean columns into integer 1 = True, 0 = False

In [5]:
data[data.select_dtypes(bool).columns] = data.select_dtypes(bool).astype(int)

data.head()

,yearConstructed,noRooms,livingSpace,noParkSpaces,balcony,hasKitchen,cellar,lift,garden,totalRent,latitude,longitude,age_of_house,price_per_m2
0,2018,3,87.00,1,1,0,1,1,0,1320.65,51.457585,7.455574,7,11.1793
1,2013,5,127.95,1,1,0,1,1,0,1878.48,51.535960,9.933196,12,10.5000
2,2018,2,47.28,1,1,0,1,1,0,624.10,53.633845,11.409783,7,11.0000
3,1930,2,44.66,2,0,0,0,0,0,340.00,51.398860,12.219155,95,5.3739
4,2000,3,82.00,2,1,0,1,0,0,960.00,50.019473,8.424950,25,7.8049


# Scale Numeric Features for optimization

In [7]:
scaler = StandardScaler()
column_to_scale = ['yearConstructed', 'noRooms', 'livingSpace','latitude', 'longitude', 'price_per_m2','age_of_house'] 
data[column_to_scale] = scaler.fit_transform(data[column_to_scale])
data.head()

,yearConstructed,noRooms,livingSpace,noParkSpaces,balcony,hasKitchen,cellar,lift,garden,totalRent,latitude,longitude,age_of_house,price_per_m2
0,0.938290,0.091652,0.004420,1,1,0,1,1,0,1320.65,-0.002189,-1.111215,-0.938290,0.334481
1,0.809488,0.759297,0.068811,1,1,0,1,1,0,1878.48,0.050401,-0.088039,-0.809488,0.229062
2,0.938290,-0.242171,-0.058037,1,1,0,1,1,0,624.10,1.458096,0.521743,-0.938290,0.306656
3,-1.328629,-0.242171,-0.062157,2,0,0,0,0,0,340.00,-0.041594,0.855986,1.328629,-0.566447
4,0.474602,0.091652,-0.003442,2,1,0,1,0,0,960.00,-0.967173,-0.710894,-0.474602,-0.189185


# Divide daten into Target value, and independent variable (Features)

In [8]:
y_data = data['totalRent']
x_data = data.drop('totalRent', axis='columns')

x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, random_state=42)

# build the different modellen for the analysis

In [9]:
rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=2)
cb = CatBoostRegressor(verbose=0)
xgb = XGBRegressor()

models = [rf, cb, xgb]

# Train the Model

In [10]:
row_list = []
for model in models:
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)

    rows = pd.DataFrame([{'Models':type(model).__name__, 'r2_score':r2_score(y_test, y_pred),\
                          'mean_absolute_error':mean_absolute_error(y_test, y_pred)}])
    row_list.append(rows)
result = pd.concat(row_list, ignore_index=True)

In [11]:
result

,Models,r2_score,mean_absolute_error
0,RandomForestRegressor,0.826183,56.135450
1,CatBoostRegressor,0.837876,50.619178
2,XGBRegressor,0.833764,55.986811


# prepare daten for deployement

In [13]:
# save the trained model
joblib.dump(cb, 'house_price_model.pkl')

# save the standerScaler
joblib.dump(scaler, 'scaler.pkl')

# save features columns
features_columns = list(x_data.columns)
joblib.dump(features_columns, 'features_columns.pkl')

['features_columns.pkl']